# F02-P2 Nature

**Site Characterization: Nature, components 2.1 and 2.2.**

Characterizes the ecological quality of the AOI forest and habitat. Where `F02-P2 General`
answers "where is this site and what has happened to it", Nature answers "how good is what is
left, and does it matter for biodiversity".

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** Complete as scoped: 2.1 FLII and 2.2 KBA.

**Known limit of that scope.** Both components need forest or a designated site to say anything. 2.1
FLII is a property of forest and returns not applicable without it, and 2.2 KBA returns a negative
sentence on most sites. A degraded grassland or a cropland targeted for planting therefore receives
almost no ecological description from this module, which is the site type where a RESTORE decision
most depends on the starting condition. Related asymmetry: the reference ecosystem has five classes
in the backend, but the only ecosystem quality metric here measures forest, so savanna and grassland
have no equivalent of FLII. Recorded as a scope limit, not as pending work.

## Handoff

Reads nothing from `F02-P2 General` at present. Both Nature components derive their own forest
extent from `forest_mask_2024`, so this notebook can run independently. It writes
`outputs/<aoi_id>__F02-P2-nature.json`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import os
import rasterio
import matplotlib.pyplot as plt

from pyproj import Geod
from rasterio.mask import mask
from rasterio.mask import raster_geometry_mask
from config import *
from common import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
aoi_id = AOI_ID   # AOI is set in config.py (AOI_PATH, AOI_ID); change it there, then restart the kernel

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}")

results: dict[str, ComponentResult] = {}

AOI aoi1: 82,263 ha


---
## 2.1 Forest Landscape Integrity (FLII)

Reports the landscape integrity of the AOI forest as a headline mean score out of 10, with a
High / Medium / Low breakdown in the narrative. Integrity is the degree to which a forest is
still intact, connected, and free of human pressure.

**Data.** `flii_mosaic_SEA_300m.tif` (continuous 0 to 10) and `flii_class_mosaic_SEA_300m.tif`
(1 = Low, 2 = Medium, 3 = High). Based on the Forest Landscape Integrity Index (Grantham et al.
2020, Nat. Commun. 11:5978), reimplemented and calibrated on the SEA data stack. Landscape
scale, native 300 m.

**Data quirk, handled in code (found 2026-08-31).** The two rasters do not carry the same mask.
The class raster uses nodata = 0, so only forest cells are valid in it. The score raster uses
nodata = -9999 and stores non-forest as a real 0.0, so it spans the whole AOI. Taking the mean
of the score raster as loaded therefore mixes non-forest zeros into what is meant to be a forest
statistic. On the Nakai-Nam Theun test AOI that pulled the headline from 5.5 down to 3.9 out of
10, while the class shares below stayed correct, so the two halves of the same card disagreed.
The code now masks the score with the class raster before averaging, and loads both onto one
grid with `like=` so the two masks line up cell for cell.

**Calibration warning.** The 0 to 10 values are calibrated on the pooled SEA distribution, so
they are not one to one with the published global FLII product. Present them as the SEA forest
integrity layer, internally consistent within this run. Do not claim absolute global integrity.
Class breaks follow the paper: High at or above 9.6, Low at or below 6.0, Medium in between.
Checked on the test AOI: the class raster gives 46.1% Low and 53.9% Medium, and the score raster
gives 47.0% below 6.0 and 53.0% between 6.0 and 9.6, so the two layers do agree on the breaks.

**Decisions locked.**

- FLII is a property of forest, so the summary covers AOI forest area only. This holds for the
  headline mean as much as for the class shares.
- Denominator = the forest extent of the class raster. Checked against the FC2024 forest used by
  1.5 and 1.6 on the test AOI: 56,737 ha here against 57,062 ha there, a 0.6% gap that comes
  from reading a 300 m layer alongside a 30 m one.
- Headline is the mean FLII score out of 10, a single big number for the frontend.
- The narrative reports the High, Medium and Low share and names the predominant class.

**Example render (predominantly High).**

> **Forest landscape integrity: 8.8 / 10**
>
> Of the forest in this area, 68% has high landscape integrity, 24% medium, and 8% low. The
> forest is predominantly high integrity, indicating largely intact and well-connected forest
> under low human pressure.

**Downstream use.** FLII is a biodiversity and ecosystem quality proxy feeding Triple Win
Pillar 1, and a pathway signal: high integrity favours PROTECT, low integrity favours RESTORE or
MANAGE. It also underpins the SCeNe high-integrity NbS criteria.

In [9]:
FLII_LOW, FLII_MEDIUM, FLII_HIGH = 1, 2, 3

FLII_GLOSS = {
    FLII_HIGH: "indicating largely intact and well-connected forest under low human pressure",
    FLII_MEDIUM: (
        "indicating moderately modified forest with some fragmentation or human pressure"
    ),
    FLII_LOW: "indicating heavily modified and fragmented forest under high human pressure",
}

# Report the non-forest cells carried by the score raster once they pass this share of the
# forest area. Purely a visibility threshold, it never changes the numbers.
FLII_NONFOREST_FLAG_PCT = 5.0


def analyze_flii(aoi: AOI) -> ComponentResult:
    """Component 2.1. Forest landscape integrity over the AOI forest."""
    # forest_mask_2024 is loaded only to catch the "no forest at all" case early, so the message
    # matches 1.5 and 1.6 rather than saying "no FLII data".
    if forest_mask_2024(aoi).is_empty:
        return not_applicable(
            "2.1 Forest Landscape Integrity",
            "No forest is present in this project area, so landscape integrity cannot be "
            "assessed.",
        )

    score = load_raster_clipped(FLII_FOREST_RASTER, aoi, resampling="bilinear")
    classes = load_raster_clipped(FLII_CLASS_RASTER, aoi, resampling="nearest", like=score)

    # The class raster IS the forest mask: its nodata is 0, so only forest cells are valid. The
    # score raster is not. It keeps -9999 for nodata and writes non-forest as a real 0.0, so it
    # spans the whole AOI. Averaging it as loaded would report an AOI mean next to a forest
    # breakdown, two different populations on one card. Mask the score with the class raster so
    # both describe the same cells. `like=` above is what makes the two masks comparable.
    # Masking by value (score == 0) would be wrong: genuinely degraded forest also scores 0.
    forest_only = np.ma.masked_array(
        np.ma.getdata(score.values),
        mask=np.ma.getmaskarray(score.values) | np.ma.getmaskarray(classes.values),
    )
    forest_score = RasterSlice(
        values=forest_only,
        pixel_area_ha=score.pixel_area_ha,
        transform=score.transform,
        crs=score.crs,
    )

    forest_area_ha = classes.valid_area_ha
    if forest_area_ha <= 0 or forest_score.valid_count == 0:
        return not_applicable(
            "2.1 Forest Landscape Integrity",
            "The forest integrity layer does not cover the forest in this project area.",
        )

    mean_flii = float(np.ma.mean(forest_score.values))   # 0 to 10, one decimal on display
    mean_flii_aoi = float(np.ma.mean(score.values))      # diagnostic, whole score extent

    rows = tabulate_classes(classes, FLII_CLASSES, denominator_ha=forest_area_ha)
    by_code = {r.code: r for r in rows}
    dom = dominant(rows)

    narrative = sentences(
        f"Of the forest in this area, {fmt_pct(by_code[FLII_HIGH].pct)} has high landscape "
        f"integrity, {fmt_pct(by_code[FLII_MEDIUM].pct)} medium, and "
        f"{fmt_pct(by_code[FLII_LOW].pct)} low.",
        f"The forest is predominantly {dom.label.lower()} integrity, {FLII_GLOSS[dom.code]}.",
    )

    flags = []
    nonforest_ha = score.valid_area_ha - forest_area_ha
    if nonforest_ha > forest_area_ha * FLII_NONFOREST_FLAG_PCT / 100:
        flags.append(
            f"The FLII score raster carries {fmt_ha(nonforest_ha)} of non-forest inside this "
            f"AOI, stored as 0. Those cells are excluded from the headline mean. Including "
            f"them would give {mean_flii_aoi:.2f} instead of {mean_flii:.2f}."
        )

    return ComponentResult(
        component="2.1 Forest Landscape Integrity",
        applicable=True,
        narrative=narrative,
        tables={"integrity": rows},
        rasters={"2.1_flii_score": forest_score, "2.1_flii_class": classes},
        values={
            "mean_flii": mean_flii,  # headline big number, forest cells only
            "dominant_class": dom.code,
            "pct_high": by_code[FLII_HIGH].pct,
            "forest_area_ha": forest_area_ha,
            # Diagnostics, not for display. Kept so the forest-only choice stays auditable.
            "mean_flii_score_extent": mean_flii_aoi,
            "score_extent_ha": score.valid_area_ha,
        },
        flags=flags,
    )


results["2.1"] = analyze_flii(aoi)
show_result(results["2.1"])

[2.1 Forest Landscape Integrity]
  Of the forest in this area, 0% has high landscape integrity, 54% medium, and 46% low. The forest is predominantly medium integrity, indicating moderately modified forest with some fragmentation or human pressure.
  integrity:


,code,label,area_ha,pct
0,1,Low,26131.026293,46.056685
1,2,Medium,30605.636816,53.943315
2,3,High,0.000000,0.000000


  saved table: D:\NBSTOOLV3\OUTPUTS\aoi1\tables\2.1_integrity.csv


{'mean_flii': 5.483228521030626,
 'dominant_class': 2,
 'pct_high': 0.0,
 'forest_area_ha': 56736.66310941539,
 'mean_flii_score_extent': 3.855513755403159,
 'score_extent_ha': 82273.40519285836}

FLAG: The FLII score raster carries 25,537 ha of non-forest inside this AOI, stored as 0. Those cells are excluded from the headline mean. Including them would give 3.86 instead of 5.48.
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\2.1_flii_score.tif
  saved raster: D:\NBSTOOLV3\OUTPUTS\aoi1\rasters\2.1_flii_class.tif


---
## 2.2 Key Biodiversity Areas (KBA)

Reports whether the AOI overlaps Key Biodiversity Areas, with overlap area, share and a
narrative.

**A KBA is not a protected area.** It is a site that contributes significantly to the global
persistence of biodiversity (IUCN KBA Standard 2016). It may or may not be legally protected.
This is a different lens from 1.3 (WDPA, legal status) and the two complement each other. The
narrative emphasises biodiversity importance, not protection.

**Data.** `KBA_polygon.shp`, World Database of Key Biodiversity Areas, BirdLife International
and the KBA Partnership.

**Decisions locked.**

- Mirrors 1.3 mechanics: headline overlap = union of KBA polygons, so overlapping or nested
  sites are not double counted. No sliver threshold, because any KBA overlap is material.
- Denominator = total AOI area. A KBA concerns the whole site, not only its forest.
- The narrative gives the KBA name only, no criteria or type.

**To verify against the actual file.** Name field assumed here: `IntName`.

**Example render.**

> This project area overlaps 210 ha (17%) of a Key Biodiversity Area, Bukit Tigapuluh. Key
> Biodiversity Areas are sites that contribute significantly to the global persistence of
> biodiversity.

**Downstream use.** Feeds Triple Win Pillar 1 and acts as a safeguard and eligibility signal. A
KBA that is not also under WDPA protection (1.3) is a biodiversity important but unprotected
site, which is a strong PROTECT and additionality rationale.

In [4]:
KBA_DEFINITION = (
    "Key Biodiversity Areas are sites that contribute significantly to the global persistence "
    "of biodiversity."
)


@dataclass(frozen=True)
class KbaSite:
    name: str
    area_ha: float


def analyze_kba(aoi: AOI) -> ComponentResult:
    """Component 2.2. Overlap with Key Biodiversity Areas."""
    gdf = load_vector_intersecting(KBA_POLYGON, aoi)

    if gdf.empty:
        return ComponentResult(
            component="2.2 Key Biodiversity Areas",
            applicable=True,  # "no overlap" is a real answer, not a missing one
            narrative="This project area does not overlap any Key Biodiversity Areas.",
            tables={"sites": []},
            values={"kba_ha": 0.0, "kba_pct": 0.0, "kba_site_count": 0},
        )

    kba_ha = union_overlap_ha(aoi, gdf)
    kba_pct = safe_pct(kba_ha, aoi.area_ha)

    areas = per_feature_overlap_ha(aoi, gdf)
    sites = sort_by_area([
        KbaSite(name=str(gdf.iloc[i].get("IntName", "Unnamed KBA")), area_ha=float(areas[i]))
        for i in range(len(gdf))
    ])

    if len(sites) == 1:
        head = (
            f"This project area overlaps {fmt_ha(kba_ha)} ({fmt_pct(kba_pct)}) of a Key "
            f"Biodiversity Area, {sites[0].name}."
        )
    else:
        largest, *rest = sites
        rest_text = oxford_join(f"{s.name} ({fmt_ha(s.area_ha)})" for s in rest)
        head = (
            f"This project area overlaps {fmt_ha(kba_ha)} ({fmt_pct(kba_pct)}) of Key "
            f"Biodiversity Areas, across {len(sites)} sites. The largest is {largest.name} "
            f"({fmt_ha(largest.area_ha)}), followed by {rest_text}."
        )

    return ComponentResult(
        component="2.2 Key Biodiversity Areas",
        applicable=True,
        narrative=sentences(head, KBA_DEFINITION),
        tables={"sites": sites},
        values={"kba_ha": kba_ha, "kba_pct": kba_pct, "kba_site_count": len(sites)},
    )


results["2.2"] = analyze_kba(aoi)
show_result(results["2.2"])

[2.2 Key Biodiversity Areas]
  This project area overlaps 14,317 ha (21%) of a Key Biodiversity Area, Gunung Niut-Poteng. Key Biodiversity Areas are sites that contribute significantly to the global persistence of biodiversity.
  sites:


,name,area_ha
0,Gunung Niut-Poteng,14316.718525


{'kba_ha': 14316.718524669994,
 'kba_pct': 21.22913156918525,
 'kba_site_count': 1}

---
## 2.3 Habitat Area

## AOH is a refinement of IUCN Range

The **AOH approach** refines coarse species range maps into spatially-explicit estimates of available suitable habitat, following the globally standardized methods of *Brooks et al., 2019*

For each species (~6,000 in total), AOH generated through the following sequential filtering steps:

1. Retain only polygons classified as **extant and native** in the IUCN/BirdLife dataset and remove polygons classified as introduced, uncertain, or extinct.

2. Extract habitat preferences from **IUCN Red List species assessment** and mask the range map to include only pixels corresponding to suitable habitat classes based on the crosswalk.

3. Apply species-specific elevational limits using a high-resolution digital elevation model. Where relevant, additional climatic constraints will be applied to refine habitat suitability for habitat types, such as savannas.

---

The resulting raster represents the intersection of **range, habitat, and environmental suitability**. Outputs will be binary (**suitable/unsuitable**) and stored as species-level rasters.

The accuracy and validity of AOH assessed using species occurrence data from the **Global Biodiversity Information Facility (GBIF)**, as well as data collected from three NbS pilot sites. Validation assess the proportion of independent occurrence records that fall within mapped AOH, providing an indication of model performance

---



# Biodiversity Habitat Intersection Analysis

This analysis identifies species with suitable habitat intersecting a user-defined Area of Interest (AOI), quantifies habitat extent, and summarizes biodiversity significance by taxonomic class, habitat overlap, and conservation status.

## Inputs

The analysis requires:

1. **User AOI polygon**
2. **Master biodiversity GeoParquet**
3. **Species habitat rasters** in `EPSG:4326`

## Analytical Workflow

The analysis follows these steps:

1. Read the AOI and calculate its geodesic area in hectares.
2. Read the master biodiversity GeoParquet.
3. Filter candidate species whose raster footprints intersect the AOI.
4. Open only habitat rasters associated with candidate species.
5. Read raster pixels located within the AOI.
6. Identify suitable habitat where `DN == 1`.
7. Calculate suitable habitat area using geodesic pixel area.
8. Calculate suitable habitat as a percentage of the AOI.
9. Classify species habitat overlap into:
   - `<20%`
   - `20–40%`
   - `40–60%`
   - `60–80%`
   - `>80%`
10. Summarize species by taxonomic class:
    - Amphibian
    - Mammal
    - Bird
    - Reptile
11. Identify species with **>90% habitat overlap** within the AOI.
12. Summarize **CR, EN, and VU** species with >90% habitat overlap.
13. Generate a biodiversity narrative summarizing the results.

## Outputs

The analysis produces:

- **AOI area** in hectares
- **Total species supported**
- **Species count by taxonomic class**
- **Species count by habitat-overlap class**
- **Species with >90% habitat overlap**
- **CR, EN, and VU species counts within the >90% overlap group**
- **Species-level habitat table**
- **Biodiversity narrative**

### Species-Level Habitat Table

The output table includes:

- `species`
- `Class`
- `redlistCategory`
- `iucn_status`
- `pattern`
- `presence`
- `origin`
- `seasonality`
- `Candidate`
- `habitat_area_ha`
- `percentage_aoi`
- `habitat_class`

In [ ]:
# =============================================================================
# HELPERS
# =============================================================================

def geometry_area_ha(geom):
    area, _ = GEOD.geometry_area_perimeter(geom)
    return abs(area) / 10000


def row_pixel_area_ha(transform, height):
    areas = np.zeros(height)

    for row in range(height):
        north = transform.f + row * transform.e
        south = north + transform.e
        west = transform.c
        east = west + transform.a

        area, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west],
            [north, north, south, south]
        )

        areas[row] = abs(area) / 10000

    return areas


def habitat_class(pct):

    if pct < 20:
        return "<20%"
    if pct < 40:
        return "20–40%"
    if pct < 60:
        return "40–60%"
    if pct <= 80:
        return "60–80%"

    return ">80%"


# =============================================================================
# READ AOI + INVENTORY
# =============================================================================

aoi = gpd.read_file(USER_POLYGON).to_crs("EPSG:4326")
aoi_geom = aoi.geometry.union_all()

aoi_area_ha = geometry_area_ha(
    aoi_geom
)

inventory = gpd.read_parquet(
    INVENTORY_PATH
).to_crs("EPSG:4326")

candidates = inventory[
    inventory.intersects(aoi_geom)
].copy()

print(
    f"Candidate species: "
    f"{len(candidates):,} / {len(inventory):,}"
)


# =============================================================================
# HABITAT ANALYSIS
# =============================================================================

results = []

for i, (_, row) in enumerate(
    candidates.iterrows(),
    start=1
):

    species = row["species"]

    print(
        f"[{i}/{len(candidates)}] {species}"
    )

    raster_path = (
        BIODIVERSITY_ROOT
        / row["raster_path"]
    )

    if not raster_path.exists():
        continue

    with rasterio.open(raster_path) as src:

        try:
            outside, transform, window = raster_geometry_mask(
                src,
                [aoi_geom.__geo_interface__],
                crop=True
            )

        except ValueError:
            continue

        data = src.read(
            1,
            window=window,
            masked=True
        )

        habitat = (
            ~outside
            & ~np.ma.getmaskarray(data)
            & (data.data == TARGET_DN)
        )

        if not habitat.any():
            continue

        row_ha = row_pixel_area_ha(
            transform,
            data.shape[0]
        )

        habitat_area_ha = float(
            np.sum(
                habitat.sum(axis=1)
                * row_ha
            )
        )

        percentage_aoi = (
            habitat_area_ha
            / aoi_area_ha
            * 100
        )

        results.append({
            "species": species,
            "Class": row["Class"],
            "redlistCategory": row["redlistCategory"],
            "iucn_status": row["iucn_status"],
            "pattern": row["pattern"],
            "presence": row["presence"],
            "origin": row["origin"],
            "seasonality": row["seasonality"],
            "Candidate": row["Candidate"],
            "habitat_area_ha": habitat_area_ha,
            "percentage_aoi": percentage_aoi,
            "habitat_class": habitat_class(
                percentage_aoi
            )
        })


# =============================================================================
# RESULT TABLE
# =============================================================================

result = pd.DataFrame(results)

if not result.empty:

    result = result.sort_values(
        ["Class", "species"]
    ).reset_index(drop=True)


# =============================================================================
# SUMMARY
# =============================================================================

classes = [
    "<20%",
    "20–40%",
    "40–60%",
    "60–80%",
    ">80%"
]

habitat_summary = (
    result["habitat_class"]
    .value_counts()
    .reindex(classes, fill_value=0)
    .to_dict()
)

species_count = len(result)

# =============================================================================
# SPECIES BY CLASS
# =============================================================================

taxon_classes = [
    "Amphibian",
    "Mammal",
    "Bird",
    "Reptile"
]

class_summary = (
    result["Class"]
    .value_counts()
    .reindex(taxon_classes, fill_value=0)
    .to_dict()
)

over_90 = result[
    result["percentage_aoi"] > 90
]

over_90_count = len(over_90)

en_count = (
    over_90["iucn_status"]
    .eq("EN")
    .sum()
)

vu_count = (
    over_90["iucn_status"]
    .eq("VU")
    .sum()
)

cr_count = (
    over_90["iucn_status"]
    .eq("CR")
    .sum()
)


# =============================================================================
# NARRATIVE
# =============================================================================

if over_90_count > 0:

    narrative = (
        f"Project area supports the area of habitat for "
        f"{species_count:,} species. A subset of "
        f"{over_90_count:,} species has more than 90% habitat "
        f"overlapped within the project, including "
        f"{en_count:,} EN, {vu_count:,} VU, and "
        f"{cr_count:,} CR species."
    )

else:

    narrative = (
        f"Project area supports the area of habitat for "
        f"{species_count:,} species. Most of the species have "
        f"less than 90% habitat overlapped within the project."
    )


# =============================================================================
# OUTPUT
# =============================================================================
print("\nSpecies by Class:")

for taxon_class in taxon_classes:

    print(
        f"{taxon_class:<12} "
        f"{class_summary[taxon_class]:>5} species"
    )

print()
print(f"AOI area: {aoi_area_ha:,.2f} ha")
print(f"Species supported: {species_count:,}")

print("\nHabitat overlap:")
for label in classes:
    print(
        f"{label:<10} "
        f"{habitat_summary[label]:>5} species"
    )

print(
    f"\nSpecies with >90% overlap: "
    f"{over_90_count:,}"
)

print()
print(narrative)

result

---
## 2.5 Charismatic species occurences

Reports the species occurence inside the AOI, with overlap point, share the quantitative data

The Charismatic species is occurence of flagship candidate species from different region accross Southeast-Asia. The data comes from GBIF (Global Biodiversity Information Facility) by compiling sources from Natural History Museums, Citizen Science Networks, Academic & Research Institutions, Government & Conservation Agencies. Following the data standards under The Darwin Core Standard (DwC), Ecological Metadata Language, and The Biological Collection Access Service.

---

The resulting shapefile represents the intersection of **species, individualCount, eventDate and basisofRecords**. Main outputs will be the total occurence (**Species Name and Record Count**)

---

## Example render

> The selected area recorded observations of **2** species.

| Occurrence summary    | Value |
| --------------------- | ----: |
| **Unique species**    | **2** |
| **Total records**     |     4 |
| **Total individuals** |     6 |

### Species Occurrence

| Species Name               | Total Occurrence | Record Count | Latest Encounter | Basis of Record   |
| -------------------------- | ---------------: | -----------: | ---------------- | ----------------- |
| `Buceros rhinoceros`       |            **6** |            3 | 23 February 2017 | HUMAN_OBSERVATION |
| `Symphalangus syndactylus` |            **0** |            1 | 03 November 2014 | HUMAN_OBSERVATION |



In [ ]:
# =============================================================================
# Helpers
# =============================================================================

def resolve_column(gdf, desired):
    """Find the actual column matching a desired name, allowing for
    shapefile 10-char truncation and case differences."""
    columns = list(gdf.columns)

    # 1. Exact match.
    if desired in columns:
        return desired

    lower_map = {col.lower(): col for col in columns}

    # 2. Case-insensitive exact match.
    if desired.lower() in lower_map:
        return lower_map[desired.lower()]

    # 3. Truncation: the shapefile field is a prefix of the desired name.
    #    Pick the longest such prefix match.
    candidates = [
        col for col in columns
        if desired.lower().startswith(col.lower()) and len(col) >= 4
    ]
    if candidates:
        return max(candidates, key=len)

    raise KeyError(
        f"Could not find a column for '{desired}'. "
        f"Available columns: {columns}"
    )


def most_frequent(series):
    modes = series.dropna().mode()
    return modes.iat[0] if not modes.empty else None


# =============================================================================
# Load inputs
# =============================================================================

points = gpd.read_file(species_points_path)
if points.empty:
    raise ValueError("The species occurrence layer contains no features.")

polygon = gpd.read_file(user_polygon_path)
if polygon.empty:
    raise ValueError("The user polygon contains no features.")

print(f"Loaded {len(points)} occurrence points.")

# Resolve the attribute columns against the actual (truncated) field names.
species_col = resolve_column(points, SPECIES_COLUMN)
count_col = resolve_column(points, COUNT_COLUMN)
date_col = resolve_column(points, DATE_COLUMN)
basis_col = resolve_column(points, BASIS_COLUMN)

print("Resolved columns:")
print(f"  species        -> {species_col}")
print(f"  individualCount-> {count_col}")
print(f"  eventDate      -> {date_col}")
print(f"  basisOfRecord  -> {basis_col}")


# =============================================================================
# Steps 1-2 - Keep points inside the polygon
# =============================================================================

# Align CRS: reproject the polygon to the points CRS.
if points.crs is None:
    raise ValueError("The occurrence layer has no CRS.")

if polygon.crs is None:
    print("Warning: polygon has no CRS; assuming it matches the points layer.")
    polygon = polygon.set_crs(points.crs)
elif polygon.crs != points.crs:
    polygon = polygon.to_crs(points.crs)

# Dissolve the polygon to a single geometry so a point can't match more than
# one polygon feature (which would double-count it).
try:
    merged_geometry = polygon.geometry.union_all()
except AttributeError:  # older geopandas
    merged_geometry = polygon.geometry.unary_union

polygon_single = gpd.GeoDataFrame(geometry=[merged_geometry], crs=polygon.crs)

clipped_points = gpd.sjoin(
    points,
    polygon_single,
    predicate=SPATIAL_PREDICATE,
    how="inner",
).drop(columns="index_right")

print(f"Occurrence points inside the polygon: {len(clipped_points)}")

if clipped_points.empty:
    raise SystemExit("No occurrence points fall inside the polygon.")


# =============================================================================
# Step 3-4 - Clean, group by species, aggregate
# =============================================================================

# individualCount -> numeric (nulls become NaN and are ignored by sum).
clipped_points[count_col] = pd.to_numeric(
    clipped_points[count_col], errors="coerce"
)

# eventDate -> datetime so "max" is chronological, not lexical.
clipped_points[date_col] = pd.to_datetime(
    clipped_points[date_col], errors="coerce"
)

# Records with no species name cannot be grouped meaningfully.
clipped_points = clipped_points.dropna(subset=[species_col])

result = (
    clipped_points
    .groupby(species_col)
    .agg(
        total_occurrence=(count_col, "sum"),      # summed individualCount
        record_count=(species_col, "count"),      # number of occurrence rows
        latest_encounter=(date_col, "max"),
        basis_of_record=(basis_col, most_frequent),
    )
    .reset_index()
    .rename(columns={species_col: "species"})
)


# =============================================================================
# Step 5 - Total unique species
# =============================================================================

total_unique_species = result["species"].nunique()


# =============================================================================
# Format output
# =============================================================================

result["total_occurrence"] = result["total_occurrence"].round().astype("Int64")
result["record_count"] = result["record_count"].astype("Int64")

result["latest_encounter"] = result["latest_encounter"].apply(
    lambda value: value.strftime("%d %B %Y") if pd.notna(value) else None
)

result = result.sort_values(
    "total_occurrence", ascending=False
).reset_index(drop=True)


# =============================================================================
# Write and display
# =============================================================================

output_path = Path(output_excel_path)
output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="species_summary", index=False)

    worksheet = writer.sheets["species_summary"]
    for column_cells in worksheet.columns:
        longest = max(
            (len(str(cell.value)) for cell in column_cells if cell.value is not None),
            default=0,
        )
        letter = column_cells[0].column_letter
        worksheet.column_dimensions[letter].width = min(longest + 2, 60)

# print()
# print(f"Total unique species: {total_unique_species}")
# print()
# print(result.to_string(index=False))
# print()
# print(f"Summary written to: {output_path}")

# ---- Notebook rendering ------------------------------------------------------
# In Jupyter / ArcGIS notebooks this shows the summary as a formatted table.
try:
    from IPython.display import display

    print()
    print(f"Total unique species: {total_unique_species}")
    display(result)
except ImportError:
    pass

# Leaving `result` as the final expression lets a notebook cell render it too.
result

---
## 2.6 Conservation Significant

NatureMap priority-rank analysis for project AOIs.

Data:  NatureMap_prioritymaps.zip (Jung et al. 2021, Nat Ecol Evol 5:1499-1509;
       Zenodo 10.5281/zenodo.5006332). Ranked priority layers only -- this archive
       contains NO raw carbon or water stock rasters.

Axes are % of AOI land area falling within the global top-X% ranked cells, one
axis per weighting scenario. This is NOT the same quantity as the paper's Fig. 1
triangle (which shows % of species targets met / % of stock conserved).

Resolution: 10 km (100 km2 per pixel), native -- nothing is resampled here.
Suitable for landscape-context statements, not for site-level or per-stratum figures.

> **AOI_A** covers **1,456 km²** and overlaps **23** valid NatureMap priority pixels.

| Area summary           | Value                       |
| ---------------------- | --------------------------: |
| **True polygon area**  |  **1,456 km²** (145,642 ha) |
| **Pixel envelope**     |      2,300 km² (230,000 ha) |
| **Envelope inflation** |                       +58 % |
| **Valid pixels**       |                          23 |

### Priority Rank by Axis

| Axis           | Min Rank | Valid Pixels |
| -------------- | :------: | -----------: |
| `Biodiversity` |  **2.0** |           23 |
| `Carbon`       |  **1.0** |           23 |
| `Water`        |  **2.0** |           23 |



In [ ]:
LAYERS = {
    "Biodiversity": r"Z:\Biodiversity\biodiversity_only_v3.tif",
    "Carbon":       r"Z:\Biodiversity\biodiversity_carbon_v3.tif",
    "Water":        r"Z:\Biodiversity\biodiversity_water_v3.tif",
}
BUDGETS = [10, 30]

AOI_NAME = "AOI_A"
AOI_SHP  = "Z:/NbS_Tools/Dummy/AOI_4326.shp"

OUT = "Z:/NbS_Tools/Dummy/output"
os.makedirs(OUT, exist_ok=True)

EARTH_R_KM = 6371.0088          # mean Earth radius

# ---------------------------------------------------------------- helpers
def pixel_area_km2(transform, shape):
    """True ground area (km2) of every cell in a 4326 raster window.

    Longitude cells are equal width everywhere; latitude bands shrink toward the
    poles. Area of a cell spanning [lat1, lat2] x d_lon degrees:
        R^2 * d_lon_rad * (sin(lat2) - sin(lat1))
    Returns a 2-D array matching `shape` (rows x cols)."""
    d_lon = abs(transform.a)                 # degrees per pixel in x
    d_lat = abs(transform.e)                 # degrees per pixel in y
    n_rows, n_cols = shape

    # latitude of each row's top and bottom edge
    top = transform.f                        # y of the upper-left corner
    row_top = top - np.arange(n_rows) * d_lat
    row_bot = row_top - d_lat
    band = (np.sin(np.radians(row_top)) - np.sin(np.radians(row_bot)))  # per row
    d_lon_rad = np.radians(d_lon)
    row_area = (EARTH_R_KM ** 2) * d_lon_rad * band          # km2 per cell, per row
    return np.repeat(row_area[:, None], n_cols, axis=1)


def clip(path, geoms):
    """Clip raster to geometry. all_touched=True keeps any pixel the polygon
    intersects. Returns (values, area_km2) as 2-D arrays with NaN outside."""
    with rasterio.open(path) as src:
        arr, transform = mask(src, geoms, crop=True, all_touched=True,
                              nodata=src.nodata, filled=True)
        vals = arr[0].astype("float64")
        if src.nodata is not None:
            vals[vals == src.nodata] = np.nan
    area = pixel_area_km2(transform, vals.shape)
    area = np.where(np.isnan(vals), np.nan, area)   # mask area outside polygon too
    return vals, area


def analyse(name, shp):
    aoi = gpd.read_file(shp)
    if aoi.crs is None:
        raise ValueError(f"{shp} has no CRS defined. If lon/lat: aoi = aoi.set_crs(4326)")
    aoi = aoi.to_crs(4326)                    # rasters are 4326; match them
    geoms = [aoi.union_all().__geo_interface__]

    # true polygon area: reproject a copy to equal-area purely for the area number
    true_km2 = aoi.to_crs("ESRI:54009").area.sum() / 1e6

    # envelope + per-pixel areas come from the first layer (all share the grid)
    ref_vals, ref_area = clip(next(iter(LAYERS.values())), geoms)
    envelope = np.nansum(ref_area)
    n_px     = int(np.sum(~np.isnan(ref_vals)))

    print(f"\n=== {name} ===")
    print(f"  true polygon area : {true_km2:,.0f} km2 ({true_km2 * 100:,.0f} ha)")
    print(f"  pixel envelope    : {envelope:,.0f} km2 ({envelope * 100:,.0f} ha)")
    if true_km2 > 0:
        print(f"  envelope inflation: {100 * envelope / true_km2 - 100:+.0f}%")
    print(f"  valid pixels      : {n_px}")
    if n_px == 0:
        raise ValueError(f"{name}: no overlap with raster grid -- check CRS/extent.")
    if n_px < 20:
        print(f"  ** WARNING: {n_px} pixels. Percentages are unstable at this count **")

    rows = []
    for label, path in LAYERS.items():
        rank, area = clip(path, geoms)
        valid = int(np.sum(~np.isnan(rank)))
        rmin  = np.nanmin(rank) if valid else np.nan
        print(f"  {label:14s} min rank {rmin:6.1f}   valid px {valid}")

        for b in BUDGETS:
            sel = rank <= b
            km2 = np.nansum(area[sel])
            rows.append({
                "aoi": name, "scenario": label, "budget_pct": b,
                "area_km2": km2, "area_ha": km2 * 100,
                "pct_of_envelope": 100 * km2 / envelope,
                "min_rank": rmin, "valid_px": valid,
                "true_km2": true_km2, "envelope_km2": envelope,
            })
    return pd.DataFrame(rows)


def radar(df, name, stem):
    axes   = list(LAYERS.keys())
    angles = np.linspace(0, 2 * np.pi, len(axes), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])
    colors = {10: "#8B1A1A", 30: "#C9A227"}

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"})
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    n_px = int(df[df.aoi == name]["valid_px"].max())

    for b in BUDGETS:
        sub  = df[(df.aoi == name) & (df.budget_pct == b)].set_index("scenario")
        vals = [sub.loc[a, "pct_of_envelope"] for a in axes]
        if sum(v > 0 for v in vals) < 3:
            print(f"  note: {name} {b}% has zero-value axes -- polygon degenerates")
        vals += vals[:1]
        ax.plot(angles, vals, color=colors[b], lw=2.5, marker="o", label=f"{b}%")
        ax.fill(angles, vals, color=colors[b], alpha=0.08)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(axes, fontsize=12)
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 50, 100])
    ax.set_yticklabels(["0%", "50%", "100%"], fontsize=9)
    ax.set_title(f"{name} — AOI area within global priority ranks\n"
                 f"NatureMap 10 km, EPSG:4326 ({n_px} pixels)", fontsize=11, pad=24)
    ax.legend(title="Global top-X% rank", loc="upper right",
              bbox_to_anchor=(1.32, 1.10), fontsize=9)

    plt.savefig(stem + ".svg", bbox_inches="tight")
    plt.savefig(stem + ".png", dpi=300, bbox_inches="tight")
    plt.show()          # renders inline in the notebook


# ---------------------------------------------------------------- run
results = analyse(AOI_NAME, AOI_SHP)
results.to_csv(f"{OUT}/aoi_priority_summary.csv", index=False)
radar(results, AOI_NAME, f"{OUT}/radar_{AOI_NAME}")

print("\n--- summary (% of pixel envelope) ---")
print(results.pivot_table(index="scenario", columns="budget_pct",
                          values="pct_of_envelope").round(1))

---
## 2.7 Plant Species Occurence

This dataset represents documented plant species occurrences across Southeast Asia, compiled from the **Global Biodiversity Information Facility (GBIF)**. Occurrence records are derived from multiple sources, including human observations, living specimens, machine observations, material citations, general occurrence records, and preserved specimens. These records provide geographical information on where tree species have been observed, collected, or documented.

The dataset is further reviewed by botanical experts to improve taxonomic consistency and the reliability of species identification.

To support biodiversity conservation screening, recorded tree species are cross-referenced with the **Convention on International Trade in Endangered Species of Wild Fauna and Flora (CITES)** to identify species listed under Appendices I, II, and III. These listings indicate the degree of international trade regulation applicable to each species.

The combined dataset supports the identification of recorded tree diversity and CITES-listed species within proposed NbS project areas, providing information relevant to biodiversity assessment, conservation planning, and the selection of appropriate NbS interventions.

In [ ]:
# Read data
aoi = gpd.read_file(USER_POLYGON)
plants = gpd.read_file(
    PLANT_GDB,
    layer=PLANT_LAYER,
    engine="pyogrio"
)

if aoi.empty or aoi.crs is None or plants.crs is None:
    raise ValueError("AOI or plant dataset has invalid/missing CRS.")


# Filter by AOI and year
aoi_geom = aoi.to_crs(plants.crs).geometry.union_all()

plants[YEAR] = pd.to_numeric(
    plants[YEAR],
    errors="coerce"
)

plants = plants[
    plants[YEAR].between(START_YEAR, END_YEAR)
    & plants.geometry.intersects(aoi_geom)
    & plants[SPECIES].notna()
].copy()


# Clean attributes
plants[SPECIES] = plants[SPECIES].astype(str).str.strip()

plants[CITES] = (
    plants[CITES]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        "APPENDIX I": "I",
        "APPENDIX II": "II",
        "APPENDIX III": "III",
        "1": "I",
        "2": "II",
        "3": "III",
    })
)


# Species summary
def get_cites(values):
    return next(
        (x for x in values if x in {"I", "II", "III"}),
        "Not listed"
    )


species_table = (
    plants.groupby(SPECIES)
    .agg(
        citesAppendix=(CITES, get_cites),
        occurrence_count=(SPECIES, "size")
    )
    .reset_index()
    .sort_values(SPECIES)
)

total_species = len(species_table)

counts = (
    species_table["citesAppendix"]
    .value_counts()
)

appendix_i = counts.get("I", 0)
appendix_ii = counts.get("II", 0)
appendix_iii = counts.get("III", 0)


# Narrative
narrative = (
    f"Based on GBIF occurrence records, the project area recorded occurrences "
    f"of {total_species:,} plant species between {START_YEAR} and {END_YEAR}. "
    f"Among these, {appendix_i:,} species are listed under CITES Appendix I, "
    f"{appendix_ii:,} under Appendix II, and {appendix_iii:,} under Appendix III."
)


# Output
print(f"""
PLANT SPECIES OCCURRENCE
------------------------
Total plant species : {total_species:,}
CITES Appendix I    : {appendix_i:,}
CITES Appendix II   : {appendix_ii:,}
CITES Appendix III  : {appendix_iii:,}

{narrative}

SPECIES LIST
------------
{species_table.to_string(index=False)}
""")

---
## Save

Each section above already ran and displayed itself. This cell writes them to one combined JSON.

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_NATURE)
print(f"Saved {path}")